# 09.11 - Structured Output

**Phase:** 09 - Generative AI

**Status:** VERIFIED

---

## 1. What Are We Solving?

Structured output constrains LLM responses to a specific format (like JSON) so they are parseable and machine-readable rather than free-form text. Most applications need structured data: JSON for APIs, tables for reports, code for execution.

## 2. Why Does This Matter?

Without structured output you must parse free-form text, which is error-prone and fragile. Structured output makes LLM integration reliable and testable.

## 3. Prerequisites

- Unit 09.10 (prompt engineering)
- Unit 09.9 (LLM APIs)

## 4. Learning Objectives

- Request JSON output
- Validate output against a schema (pydantic)
- Compare JSON mode vs function calling
- Implement retry and repair logic for malformed output

## 5. Mental Model

Structured output is like filling out a form instead of writing an essay: the model is constrained to fill specific fields with specific types, making the output predictable and parseable.

```text
define JSON schema -> prompt/constrain model -> parse -> validate -> retry/repair on failure
```


## 6. Setup & Mock LLM

No API key is available. We mock the model to return JSON and focus on parsing, validation, and repair - the parts you control.


In [1]:
import matplotlib
matplotlib.use('Agg')
import json
from pydantic import BaseModel, ValidationError
print("pydantic ready.")


pydantic ready.


## 7. Define a JSON Schema (pydantic model)

Declare the expected fields and types. This is both the prompt contract and the validation target.


In [2]:
class Person(BaseModel):
    name: str
    age: int
    city: str

# Show the JSON schema
print(json.dumps(Person.model_json_schema(), indent=2))


{
  "properties": {
    "name": {
      "title": "Name",
      "type": "string"
    },
    "age": {
      "title": "Age",
      "type": "integer"
    },
    "city": {
      "title": "City",
      "type": "string"
    }
  },
  "required": [
    "name",
    "age",
    "city"
  ],
  "title": "Person",
  "type": "object"
}


## 8. Mock Model Returning JSON

Simulate the model producing JSON with `response_format={"type": "json_object"}` (JSON mode) or via function calling. Here we return a canned valid JSON string.


In [3]:
def mock_extract_json(prompt):
    # In production (JSON mode):
    # resp = client.chat.completions.create(model=..., messages=[...], response_format={"type": "json_object"})
    # content = resp.choices[0].message.content
    content = '{"name": "John", "age": 30, "city": "New York"}'
    return content

raw = mock_extract_json("Extract person info from text")
print("Raw model output:", raw)

parsed = json.loads(raw)
person = Person(**parsed)
print("Validated Person:", person)
print("age type:", type(person.age).__name__)


Raw model output: {"name": "John", "age": 30, "city": "New York"}
Validated Person: name='John' age=30 city='New York'
age type: int


## 9. Validation Catches Bad Output

Even with JSON mode, models sometimes return wrong types or missing fields. pydantic raises a validation error we must handle.


In [4]:
bad = '{"name": "Alice", "age": "thirty", "city": "Paris"}'   # age is a string, not int
try:
    p = Person(**json.loads(bad))
    print("Parsed:", p)
except ValidationError as e:
    print("Validation failed:")
    for err in e.errors():
        print(f"  field={err['loc']} problem={err['msg']}")
print("\nAlways validate - don't trust that JSON has the right types.")


Validation failed:
  field=('age',) problem=Input should be a valid integer, unable to parse string as an integer

Always validate - don't trust that JSON has the right types.


## 10. Retry & Repair

Production pattern: on parse/validation failure, retry with a repair prompt that includes the error. We simulate a flaky model.


In [5]:
attempts = [
    '{name: John, age: 30, city: New York}',      # invalid JSON
    '{"name": "John", "age": 30, "city": "New York"}',  # valid
]

def robust_extract(prompt, max_retries=3):
    for attempt in range(max_retries):
        raw = attempts[attempt] if attempt < len(attempts) else '{}{}valid'
        try:
            data = json.loads(raw)
            person = Person(**data)
            return person
        except (json.JSONDecodeError, ValidationError) as ex:
            print(f"  attempt {attempt}: failed ({type(ex).__name__}) - retrying with repair instruction")
    raise RuntimeError("Could not produce valid output after retries")

result = robust_extract("Extract person info")
print("Recovered:", result)


  attempt 0: failed (JSONDecodeError) - retrying with repair instruction
Recovered: name='John' age=30 city='New York'


## 11. Function Calling for Structured Output

An alternative to JSON mode: define a tool/function whose parameters are the schema. The model returns arguments that must match. We mock the structured tool-call result.


In [6]:
def mock_function_call(prompt):
    # In production the model returns a tool_call with function.arguments (a JSON string)
    return json.dumps({"name": "Jane", "age": 25, "city": "London"})

args_str = mock_function_call("Extract from: 'Jane is 25 and lives in London.'")
print("Function arguments:", args_str)
person2 = Person(**json.loads(args_str))
print("Validated Person:", person2)
print("\nFunction calling enforces the schema server-side more strictly than JSON mode.")


Function arguments: {"name": "Jane", "age": 25, "city": "London"}
Validated Person: name='Jane' age=25 city='London'

Function calling enforces the schema server-side more strictly than JSON mode.


## 12. Handling Partial Extraction

Some fields may be missing or unknown. A robust pipeline handles optional fields.


In [7]:
class Product(BaseModel):
    name: str
    price: float
    category: str | None = None     # optional field

partial = '{"name": "Widget", "price": 9.99}'
p = Product(**json.loads(partial))
print("Product with missing optional field:", p)
print("category defaulted to:", p.category)


Product with missing optional field:

 name='Widget' price=9.99 category=None
category defaulted to: None


## 13. Debugging & Best Practices

| Symptom | Cause | Fix |
|---|---|---|
| JSON parse error | invalid JSON from model | retry with repair prompt |
| Missing required fields | schema too complex | simplify / split calls |
| Wrong field types | schema unclear | add descriptions / enum |
| Output not matching schema | model ignored schema | constrained decoding / function calling |

- Always validate even with JSON mode.
- Start with simple schemas; add complexity gradually.
- Include field descriptions in the schema.
- Implement retry + repair for parse failures.

## 14. Common Mistakes

- Not validating output (malformed JSON crashes downstream).
- Overly complex schemas.
- No retry logic.
- Assuming output is always valid.

## 15. When NOT to Use Structured Output

- Free-form creative text (no schema).
- Quick prototypes where parsing is overkill.
- Very complex nested schemas -> split into multiple calls.

## 16. Challenge

Write a repair that takes a malformed JSON string (missing closing brace) and fixes it by appending the brace, then validates.


In [8]:
def repair_json(s):
    for i in range(len(s) + 1):
        candidate = s + '}' * (i + 1)
        try:
            return json.loads(candidate)
        except json.JSONDecodeError:
            continue
    raise ValueError("Cannot repair")

broken = '{"name": "Zoe", "age": 40, "city": "Berlin"'   # missing closing brace
fixed = repair_json(broken)
print("Repaired:", fixed)
print("Valid Person:", Person(**fixed))
print("\nSimple repair strategies fix common malformed-JSON failures.")


Repaired: {'name': 'Zoe', 'age': 40, 'city': 'Berlin'}
Valid Person: name='Zoe' age=40 city='Berlin'

Simple repair strategies fix common malformed-JSON failures.


## 17. Closed-Book Recall

1. What is the difference between JSON mode and function calling for structured output?
2. Why should you always validate output even with schema enforcement?
3. How do you handle partial extraction results?
4. What is constrained decoding?

## 18. Teach-Back Questions

Explain to another person:

- Why validation matters even when the model is told to output JSON.
- How a retry-with-repair loop works.

## 19. Summary

You defined a pydantic schema, got JSON from a mocked model, validated it, implemented retry/repair, used function-calling style extraction, and handled optional/partial fields.

## 20. Further Experiment

- Add more repair strategies (fix quotes, types) and compare recovery rates.
- With a real key, compare JSON mode vs function calling reliability on the same task.

## 21. Verification Status

```
STATUS: VERIFIED
EXECUTION: PASS
DEPENDENCIES: pydantic, json
OUTPUTS: PASS
LAST VERIFIED: 2026-08-29
```
